### System Integration Test (Loading Models & Data)

In [ ]:
import pandas as pd
import pickle
import os
import gc

# 1. Verify paths to assets
data_path = '../data/processed/cleaned_data.csv'
success_model_path = '../models/success_model.pkl'
funding_model_path = '../models/funding_model.pkl'

paths_to_verify = [data_path, success_model_path, funding_model_path]
for path in paths_to_verify:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required project asset: {path}")

# 2. Free up memory before loading heavy objects
gc.collect()

# 3. Load processed features
print("🔄 Loading dataset...")
df = pd.read_csv(data_path)

# 4. Load models with process logging
print("🔄 Loading classification model (success_model)... Please wait.")
with open(success_model_path, 'rb') as f:
    success_model = pickle.load(f)

print("🔄 Loading forecasting model (funding_model)... Please wait.")
with open(funding_model_path, 'rb') as f:
    funding_model = pickle.load(f)

print("\n🎉 System Health Check: PASS!")
print(f"Dataset Shape: {df.shape}")
print("Both Machine Learning models successfully loaded and mounted in RAM!")

### Live Prediction Simulation (Mock Query Test)

In [ ]:
# 1. Get the exact list of features the models expect
success_features = success_model.feature_names_in_
funding_features = funding_model.feature_names_in_

# 2. Prepare the sample row and align dummy variables
sample_row = df.iloc[[0]].copy()

# Perform get_dummies on the sample row
sample_encoded = pd.get_dummies(sample_row, columns=['country_code_cleaned', 'Industry_Sector_cleaned'])

# Align with success classifier features (fill missing with 0)
X_sample_success = pd.DataFrame(index=[0])
for col in success_features:
    if col in sample_encoded.columns:
        X_sample_success[col] = sample_encoded[col].values
    else:
        X_sample_success[col] = 0

# Align with funding forecaster features (fill missing with 0)
X_sample_funding = pd.DataFrame(index=[0])
for col in funding_features:
    if col in sample_encoded.columns:
        X_sample_funding[col] = sample_encoded[col].values
    else:
        X_sample_funding[col] = 0

# 3. Run real-time simulation queries across both models
predicted_success = success_model.predict(X_sample_success)[0]
predicted_success_proba = success_model.predict_proba(X_sample_success)[0][1]
predicted_log_funding = funding_model.predict(X_sample_funding)[0]

# 4. Print out the diagnostic simulation results
print(f"--- Diagnostic Run for Startup: '{df.iloc[0]['Startup_Name']}' ---")
print(f"Current Operational Status: {df.iloc[0]['Startup_Status']}")
print(f"Model Success Prediction: {'Highly Probable Exit (1)' if predicted_success == 1 else 'Operating/Closed (0)'}")
print(f"Calculated Success Probability Score: {predicted_success_proba * 100:.2f}%")
print(f"Predicted Valuation Scale (Log Funding): {predicted_log_funding:.2f}")